# <strong> Pharmaceutical Research Assistant Agent: </strong>

This demonstrates how to build an intelligent pharmaceutical research assistant using LangChain’s The agent simulates real-world pharmaceutical workflows and provides reliable, domain-specific assistance.

## Key Features:

Drug Information Lookup: Retrieve details such as usage and contraindications from a mock database.

Safety Checks: Verify contraindications and ensure patient safety.

Dosage Calculations: Compute recommended dosages based on patient weight and prescribed mg/kg.

Medical Knowledge Retrieval: Search external medical sources (simulated PubMed API) for recent research or studies.

Dynamic Tool Selection: The agent automatically selects the most relevant tool based on user input.

## Technical Highlights:

@tool Decorators: Define modular, callable functions that the agent can invoke.

Connects the language model with tools, enabling reasoning and tool selection.

Language Model Backend: Uses ChatOpenAI with the gpt-4o-mini model for accurate and context-aware responses.

System Prompt Guidance: Ensures the agent behaves safely, uses tools when appropriate, and avoids hallucinating medical information.

## Workflow:

User Input: The user queries the agent (e.g., “Calculate dosage for a 70kg patient at 10 mg/kg”).

Tool Selection: The agent identifies the most relevant tool automatically.

Tool Invocation: Executes the selected tool and retrieves results.

Response Generation: Combines the tool output with explanatory context if requested.

## Benefits:

Provides domain-specific intelligence for pharmaceutical research and clinical workflows.

Ensures safety-first outputs with explicit use of tools.

Highly extensible — can integrate real databases, APIs, and multi-step workflows.

Facilitates automation in research, drug safety evaluation, and patient support systems.

In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
import utils

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
mock_database = {
    "aspirin": {"use": "Pain relief", "contraindications": "Ulcers, bleeding disorders"},
    "metformin": {"use": "Diabetes Type 2", "contraindications": "Severe kidney issues"},
    "ibuprofen": {"use": "Inflammation and fever", "contraindications": "Asthma, bleeding disorders"},
    "paracetamol": {"use": "Pain relief and fever", "contraindications": "Liver disease"},
    "amoxicillin": {"use": "Bacterial infections", "contraindications": "Penicillin allergy"},
    "atorvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "lisinopril": {"use": "Hypertension", "contraindications": "Pregnancy, angioedema history"},
    "omeprazole": {"use": "Acid reflux", "contraindications": "Liver disease"},
    "simvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "levothyroxine": {"use": "Hypothyroidism", "contraindications": "Untreated adrenal insufficiency"},
    "ciprofloxacin": {"use": "Bacterial infections", "contraindications": "Tendon disorders"},
    "clopidogrel": {"use": "Prevent blood clots", "contraindications": "Active bleeding"},
    "fluoxetine": {"use": "Depression", "contraindications": "MAOI use"},
    "warfarin": {"use": "Prevent blood clots", "contraindications": "Bleeding disorders"}
}

In [ ]:
@tool
def lookup_drug(drug_name: str) -> str:
    """this would look up drug information from a database"""
    drug = mock_database.get(drug_name.lower())
    return drug if drug else "Drug not found."

@tool
def safety_check(drug_name: str) -> str:
    """this would check for contraindications"""
    drug = mock_database.get(drug_name.lower())
    if not drug:
        return "No safety data available."
    return f"Contraindications: {drug['contraindications']}"

@tool
def dosage_calculator(input_data: str) -> str:
    """
    Example: 'patient_weight=75 mg_per_kg=5'
    """
    values = dict(item.split("=") for item in input_data.split())
    dose = float(values["patient_weight"]) * float(values["mg_per_kg"])
    return f"Recommended dose: {dose} mg"

@tool
def external_medical_search(query: str) -> str:
    """this would call an external medical database API"""
    return f"(Simulated summary from PubMed/clinical API) About: {query}"
    # In real implementation, integrate with clinical trial APIs


tools = [lookup_drug, safety_check, dosage_calculator, external_medical_search]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert pharmaceutical assistant. Use the tools provided to answer pharmaceutical-related queries accurately."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
output = executor.invoke({"input": "Provide information on ibuprofen, check its safety for a patient with asthma, calculate dosage for a 70kg patient at 10mg/kg, and summarize recent studies on its efficacy."})

In [ ]:
output = executor.invoke({"input": "What are the uses and contraindications of metformin?"})